# NEWFIRM target submission (list) to the AEON queue
Tomas Ahumada - tomas.ahumada@noirlab.edu

Last modified: Aug 2026


This code intends to be a tutrorial for NOIRLab-AEON users of the NOAO Extremely Wide Field Infrared Imager (NEWFIRM) mounted at the 4m V. M. Blanco telescope. The AEON queue is run by the Las Cumbres Observatory (LCO) Scheduler, thus the user requires an active account to access the LCO portal and generate a LCO key to submit requests to an active program.

Information about NEWFIRM can be found here: https://noirlab.edu/science/programs/ctio/instruments/newfirm

Information about LCO can be found here: https://observe.lco.global/

Once you have an active user in the LCO portal, you can find the API key here https://observe.lco.global/accounts/profile

# Outline
1. Read example target list
2. Get template request (json format) - this json file is modified and later sent to the LCO queue
3. Make the payload, modifying the json template.
4. Submit targets individually. Each target can have multiple filters.

# New Section

In [1]:
import io
import json
import time
import urllib.request
from urllib.parse import urlparse

from astropy.time import Time
import numpy as np
import pandas as pd
import requests

In [2]:
# get targets
raw_url = "https://raw.githubusercontent.com/tahumada/MSO-AEON/main/NEWFIRM/example_target_list"

# Append a cache-buster parameter using the current time
url_no_cache = f"{raw_url}?v={int(time.time())}"

# Read directly into pandas
targets = pd.read_csv(url_no_cache)

targets


,sourceID,ra,dec,filters,dither,sequences
0,ZTF25acfxvcu,0.646663,-3.710376,JHK,2x2,1
1,ZTF24aatlsjq,1.277942,29.962589,JHK,5-point,2
2,ZTF25abqvssg,2.797111,1.804104,JHK,3x3,1
3,ZTF24aaymkrs,3.089388,31.063407,JHK,4x4,1
4,ZTF24aawklme,3.090119,17.794190,J,5-point,3
5,ZTF25abjjlgl,5.021151,8.511636,HK,5-point,1
6,ZTF24aboafrj,5.466447,9.255386,JK,5-point,1
7,ZTF25aaqqjud,5.723418,46.144067,JH,5-point,1


In [3]:
# get example json

def get_example(example_path):
    # Automatically convert standard GitHub file URLs to raw GitHub URLs
    if "github.com" in example_path and "/blob/" in example_path:
        example_path = example_path.replace("github.com", "raw.githubusercontent.com").replace("/blob/", "/")

    # Check if the string is a URL
    if urlparse(example_path).scheme in ('http', 'https'):
        response = requests.get(example_path)

        if response.status_code == 200:
            return response.json()
        else:
            raise Exception(f"Failed to fetch file from URL. Status code: {response.status_code}")
    else:
        # Code to handle local file paths...
        pass



In [28]:
template_json_url = "https://raw.githubusercontent.com/tahumada/newfirm-tda-tools/main/example.json"
get_example(template_json_url)

{'name': 'test',
 'proposal': '2025B-716366',
 'ipp_value': 1.05,
 'operator': 'SINGLE',
 'observation_type': 'NORMAL',
 'requests': [{'acceptability_threshold': 90,
   'configuration_repeats': 1,
   'optimization_type': 'TIME',
   'configurations': [{'type': 'EXPOSE',
     'instrument_type': 'BLANCO_NEWFIRM',
     'extra_params': {'dither_value': 80,
      'dither_sequence': '5-point',
      'detector_centering': 'det_1',
      'dither_sequence_random_offset': True},
     'instrument_configs': [{'exposure_count': 1,
       'exposure_time': '20',
       'mode': 'fowler1',
       'rotator_mode': '',
       'extra_params': {'coadds': '2',
        'sequence_repeats': 1,
        'offset_ra': 0,
        'offset_dec': 0},
       'optical_elements': {'filter': 'jx'}},
      {'exposure_count': 1,
       'exposure_time': '10',
       'mode': 'fowler1',
       'rotator_mode': '',
       'extra_params': {'coadds': '3',
        'sequence_repeats': 1,
        'offset_ra': 0,
        'offset_dec': 0

In [23]:
# make payload
def make_payload(variables, template_path):
    data = get_example(template_path)

    JX = data['requests'][0]['configurations'][0]['instrument_configs'][0]
    HX = data['requests'][0]['configurations'][0]['instrument_configs'][1]
    KX = data['requests'][0]['configurations'][0]['instrument_configs'][2]

    maximum_airmass = variables['maximum_airmass']
    minimum_lunar_distance = variables['minimum_lunar_distance']

    # defaults
    data['requests'][0]['configurations'][0]['constraints']['max_airmass'] = maximum_airmass
    data['requests'][0]['configurations'][0]['constraints']['minimum_lunar_distance'] = minimum_lunar_distance

    # variables
    date = str(Time.now().value.year)+str(Time.now().value.month)+str(Time.now().value.day)
    data['name'] = variables['target']['id']+'_'+date
    data['proposal'] = variables['proposal']
    data['requests'][0]['windows'] = variables['windows']
    data['requests'][0]['configurations'][0]['target']['name'] = variables['target']['id']
    data['requests'][0]['configurations'][0]['target']['ra'] = str(variables['target']['ra'])
    data['requests'][0]['configurations'][0]['target']['dec'] = str(variables['target']['dec'])
    data['requests'][0]['configurations'][0]['extra_params']['dither_sequence'] = variables['dither_sequence']

    # reset filter configuration
    data['requests'][0]['configurations'][0]['instrument_configs'] = []

    # adding the filters
    if 'J' in variables['target']['filters']:
        JX['extra_params']['sequence_repeats'] = variables['sequence_repeats']
        data['requests'][0]['configurations'][0]['instrument_configs'].append(JX)
    if 'H' in variables['target']['filters']:
        HX['extra_params']['sequence_repeats'] = variables['sequence_repeats']
        data['requests'][0]['configurations'][0]['instrument_configs'].append(HX)
    if 'K' in variables['target']['filters']:
        KX['extra_params']['sequence_repeats'] = variables['sequence_repeats']
        data['requests'][0]['configurations'][0]['instrument_configs'].append(KX)

    return data

In [24]:
LCO_TOKEN ='HERE YOUR LCO TOKEN'
requestpath  = "https://observe.lco.global/api/requestgroups/"

PROPOSAL_ID  = 'HERE YOUR PROP_ID'
WINDOW_START = '2026-08-01 17:32:00'
WINDOW_END   = '2026-09-01 17:32:00'


lco_sent,lco_failed = [],[]
send = True

for i in range(min(2, len(targets))):

    NAME, RA, DEC, FILTERS, DITHER_SEQUENCE, SEQUENCE_REPEAT = targets.iloc[i][['sourceID', 'ra', 'dec', 'filters', 'dither', 'sequences']]

    variables = {
        "proposal": PROPOSAL_ID,
        'sequence_repeats': int(SEQUENCE_REPEAT),
        'maximum_airmass': 1.4,
        'minimum_lunar_distance': 20,
        'dither_sequence': DITHER_SEQUENCE,
        'windows': [{'start': WINDOW_START, 'end': WINDOW_END}],
        'target': {'id': NAME,
                   'ra': RA,
                   'dec': DEC,
                   'filters': FILTERS}
    }

    data = make_payload(variables, template_path = template_json_url)
    print(data)

    # send as test
    if send:
      response = requests.post(
              requestpath,
              headers={"Authorization": f"Token {LCO_TOKEN}"},
              json=data,  # Make sure you use json!
          )

      if response.status_code == 400:
          print(variables['target']['id'], 'Failed (sending to queue)')
          lco_failed.append([variables['target']['id'],response.text])

      elif response.status_code == 201 or response.status_code == 200:
          lco_sent.append([variables['target']['id'],'sent!',response.json()['id']])



{'name': 'ZTF25acfxvcu_2026811', 'proposal': 'newfirm_default', 'ipp_value': 1.05, 'operator': 'SINGLE', 'observation_type': 'NORMAL', 'requests': [{'acceptability_threshold': 90, 'configuration_repeats': 1, 'optimization_type': 'TIME', 'configurations': [{'type': 'EXPOSE', 'instrument_type': 'BLANCO_NEWFIRM', 'extra_params': {'dither_value': 80, 'dither_sequence': '2x2', 'detector_centering': 'det_1', 'dither_sequence_random_offset': True}, 'instrument_configs': [{'exposure_count': 1, 'exposure_time': '20', 'mode': 'fowler1', 'rotator_mode': '', 'extra_params': {'coadds': '2', 'sequence_repeats': 1, 'offset_ra': 0, 'offset_dec': 0}, 'optical_elements': {'filter': 'jx'}}, {'exposure_count': 1, 'exposure_time': '10', 'mode': 'fowler1', 'rotator_mode': '', 'extra_params': {'coadds': '3', 'sequence_repeats': 1, 'offset_ra': 0, 'offset_dec': 0}, 'optical_elements': {'filter': 'hx'}}, {'exposure_count': 1, 'exposure_time': '10', 'mode': 'fowler1', 'rotator_mode': '', 'extra_params': {'coadd

In [25]:
print('sources sent:', lco_sent)


sources sent: [['ZTF25acfxvcu', 'sent!', 2645084]]


In [26]:
print('sources failed:',lco_failed)

sources failed: [['ZTF24aatlsjq', '{"requests":[{"non_field_errors":["According to the constraints of the request, the target will not be visible within the time window. Check that the target is in the nighttime sky. Consider modifying the time window or loosening the airmass or lunar separation constraints. If the target is non sidereal, double check that the provided elements are correct."]}]}']]


In [27]:
ids = np.asarray(lco_sent).T[-1]
print(ids)
for idlco in ids:
    response = requests.post(
                f'https://observe.lco.global/api/requestgroups/{idlco}/cancel/',
                headers={"Authorization": f"Token {LCO_TOKEN}"},
            )

    if response.status_code == 200:
      print('request:',idlco,'cancelled')

['2645084']
request: 2645084 cancelled
